In [ ]:
!pip install python-docx
!pip install pymupdf
!pip install pdfplumber
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 80.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 101.8 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import pandas as pd
from docx import Document

In [ ]:
BASE_PATH = "/content/drive/MyDrive/AI-Placement-Copilot-ML"

resume_path = os.path.join(BASE_PATH, "datasets/resumes")
processed_path = os.path.join(BASE_PATH, "datasets/processed")

print("Resume Folder:", resume_path)
print("Processed Folder:", processed_path)

Resume Folder: /content/drive/MyDrive/AI-Placement-Copilot-ML/datasets/resumes
Processed Folder: /content/drive/MyDrive/AI-Placement-Copilot-ML/datasets/processed


In [ ]:
files = os.listdir(resume_path)

print("Total resumes:", len(files))

Total resumes: 228


In [ ]:
from docx import Document

def read_docx(file_path):
    doc = Document(file_path)
    text = []

    for para in doc.paragraphs:
        text.append(para.text)

    return "\n".join(text)

In [ ]:
sample_file = os.path.join(resume_path, files[0])

text = read_docx(sample_file)

print(text[:1000])

KETHANA Y
Role: BUSINESS ANALYST/PROJECT MANAGER
CERTIFIED SCRUM MASTER-License 752763
Email: ajay@appsboat.com		                                                         Phone: 214-501-6784

PROFESSIONAL SUMMARY:
A techno-savvy Business Analyst with out-of- the-box thinking and 5 successful years of experience in Business Analysis, Business Modeling, Requirement Gathering, Technical Documentation and Data Analysis. Domain experience comprising Healthcare, Banking with strong expertise in all phases of the Software Development Life Cycle and a firm understanding of the technologies involved. 
Experience in interacting with business teams, programmers and technical staff at all levels. 
Strong ability to elicit, document, validate and transform business requirements at various stages of software development life cycle (SDLC).
Extensive success in translating business requirements and user expectations into detailed specifications employing Unified Modeling Language (UML).
Strong business

In [ ]:
resume_texts = []

for file in files:
    if file.endswith(".docx"):
        path = os.path.join(resume_path, file)

        try:
            text = read_docx(path)

            resume_texts.append({
                "filename": file,
                "text": text
            })

        except Exception as e:
            print(file, e)

print("Extracted:", len(resume_texts))

Extracted: 224


In [ ]:
import pandas as pd

df = pd.DataFrame(resume_texts)

df.head()

,filename,text
0,KY BA PM UPDATED RESUME .docx,KETHANA Y\nRole: BUSINESS ANALYST/PROJECT MANA...
1,Adhi Gopalam - SM.docx,Adhi Gopalam\nadhigopalam@gmail.com\n281-212-3...
2,Kashyap K. Vora resume.docx,Kashyap K. Vora\nPhone: 201-532-6397. Email: k...
3,Ami Jape.docx,"Ami Jape\n50 GRAHAM ST, JERSEY CITY NJ 07307\n..."
4,Balakrishna Sudabathula.docx,Name: Balakrishna Sudabathula\n Email: bsuda...


In [ ]:
df.to_csv(
    os.path.join(processed_path, "resumes.csv"),
    index=False
)

print("Raw resumes saved successfully!")

Raw resumes saved successfully!


In [ ]:
import re

def clean_text(text):
    text = str(text)

    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\t+', ' ', text)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\S+', '', text)

    return text.strip()

In [ ]:
df["clean_text"] = df["text"].apply(clean_text)

df[["filename", "clean_text"]].head()

,filename,clean_text
0,KY BA PM UPDATED RESUME .docx,KETHANA Y\nRole: BUSINESS ANALYST/PROJECT MANA...
1,Adhi Gopalam - SM.docx,Adhi Gopalam\nadhigopalam@gmail.com\n281-212-3...
2,Kashyap K. Vora resume.docx,Kashyap K. Vora\nPhone: 201-532-6397. Email: k...
3,Ami Jape.docx,"Ami Jape\n50 GRAHAM ST, JERSEY CITY NJ 07307\n..."
4,Balakrishna Sudabathula.docx,Name: Balakrishna Sudabathula\n Email: bsudaba...


In [ ]:
def extract_email(text):
    emails = re.findall(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    return emails[0] if emails else ""

def extract_phone(text):
    phones = re.findall(r'\+?\d[\d\s\-()]{8,15}', text)
    return phones[0] if phones else ""

In [ ]:
df["email"] = df["clean_text"].apply(extract_email)
df["phone"] = df["clean_text"].apply(extract_phone)

df[["filename", "email", "phone"]].head()

,filename,email,phone
0,KY BA PM UPDATED RESUME .docx,ajay@appsboat.com,214-501-6784\n
1,Adhi Gopalam - SM.docx,adhigopalam@gmail.com,281-212-3592\n
2,Kashyap K. Vora resume.docx,kashyapkvora@gmail.com,201-532-6397
3,Ami Jape.docx,Ami.ITManager@gmail.com,
4,Balakrishna Sudabathula.docx,bsudabathula@gmail.com,669) 261-3506 \n


In [ ]:
skills = [
    "Python","Java","C++","C","SQL",
    "JavaScript","React","Node.js",
    "MongoDB","MySQL","HTML","CSS",
    "Machine Learning","Deep Learning",
    "TensorFlow","PyTorch",
    "Flask","FastAPI",
    "Docker","Kubernetes",
    "AWS","Azure","Git","Linux"
]

In [ ]:
def extract_skills(text):
    found = []

    lower = text.lower()

    for skill in skills:
        if skill.lower() in lower:
            found.append(skill)

    return list(set(found))

In [ ]:
df["skills"] = df["clean_text"].apply(extract_skills)

df[["filename", "skills"]].head()

,filename,skills
0,KY BA PM UPDATED RESUME .docx,"[C, SQL]"
1,Adhi Gopalam - SM.docx,"[Java, C]"
2,Kashyap K. Vora resume.docx,"[C, Python, Azure, C++, SQL, HTML, AWS, CSS, G..."
3,Ami Jape.docx,"[C, SQL, Git]"
4,Balakrishna Sudabathula.docx,"[C, SQL, JavaScript, HTML, AWS, CSS, Git, Linu..."


In [ ]:
df.to_csv(
    os.path.join(processed_path, "processed_resumes.csv"),
    index=False
)

print("Processed resume dataset saved successfully!")

Processed resume dataset saved successfully!
